In [4]:
#loading the featured dataset for model training
import pandas as pd
df = pd.read_pickle(r"C:\Users\Rishj\OneDrive\Desktop\PROJECTS\supply_chain_intelligence\data\processed\final_dataset_for_model_training.pkl")
print(df.shape)

(172762, 60)


In [5]:
from sklearn.model_selection import train_test_split

feature_cols = ['mode_avg_delay', 'mode_delay_std', 'mode_reliability_score',
                 'region_avg_delay', 'region_delay_std', 'region_reliability_score',
                 'mode_First Class', 'mode_Same Day', 'mode_Second Class', 'mode_Standard Class']

X = df[feature_cols]
y = df['Late_delivery_risk']

split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)


(138209, 10) (34553, 10)
(138209,) (34553,)


In [6]:
#building a Logistic Regression model
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000)
model.fit(X_train , y_train)
y_pred=model.predict(X_test)

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[13395  1362]
 [ 9142 10654]]
              precision    recall  f1-score   support

           0       0.59      0.91      0.72     14757
           1       0.89      0.54      0.67     19796

    accuracy                           0.70     34553
   macro avg       0.74      0.72      0.69     34553
weighted avg       0.76      0.70      0.69     34553



In [8]:
#building a Random Forest Classifier model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [9]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[ 7193  7564]
 [ 4963 14833]]
              precision    recall  f1-score   support

           0       0.59      0.49      0.53     14757
           1       0.66      0.75      0.70     19796

    accuracy                           0.64     34553
   macro avg       0.63      0.62      0.62     34553
weighted avg       0.63      0.64      0.63     34553



In [10]:
from sklearn.metrics import accuracy_score
train_pred = rf_model.predict(X_train)
print("Train accuracy:", accuracy_score(y_train, train_pred))
print("Test accuracy:", accuracy_score(y_test, y_pred_rf))

Train accuracy: 0.9999710583247111
Test accuracy: 0.6374555031401036


In [11]:
#checking the accuracy of the Logistic Regression model on training and test data
from sklearn.metrics import accuracy_score

# Predictions on training data
y_train_pred_lr = model.predict(X_train)

# Predictions on test data (you already have this as y_pred)
train_accuracy_lr = accuracy_score(y_train, y_train_pred_lr)
test_accuracy_lr = accuracy_score(y_test, y_pred)

print("Logistic Regression - Train accuracy:", train_accuracy_lr)
print("Logistic Regression - Test accuracy:", test_accuracy_lr)

Logistic Regression - Train accuracy: 0.6977765557959322
Logistic Regression - Test accuracy: 0.6960032413972738


In [12]:
#Fix: constrain the tree growth so it can't memorize, only generalize. Retrain with limits
rf_model_tuned = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,          # limits how deep each tree can grow
    min_samples_leaf=50,  # requires at least 50 samples per leaf (prevents tiny, overfit splits)
    random_state=42,
    n_jobs=-1
)
rf_model_tuned.fit(X_train, y_train)

# Check train vs test accuracy again
train_pred_tuned = rf_model_tuned.predict(X_train)
test_pred_tuned = rf_model_tuned.predict(X_test)

print("Train accuracy:", accuracy_score(y_train, train_pred_tuned))
print("Test accuracy:", accuracy_score(y_test, test_pred_tuned))

Train accuracy: 0.7028413489714852
Test accuracy: 0.6962926518681446


In [13]:
print(confusion_matrix(y_test, test_pred_tuned))
print(classification_report(y_test, test_pred_tuned))

[[13178  1579]
 [ 8915 10881]]
              precision    recall  f1-score   support

           0       0.60      0.89      0.72     14757
           1       0.87      0.55      0.67     19796

    accuracy                           0.70     34553
   macro avg       0.73      0.72      0.69     34553
weighted avg       0.76      0.70      0.69     34553



In [14]:
#saving the tuned Random Forest model to a file for later use(rl tuned model)
import joblib
joblib.dump(rf_model_tuned, 'delay_prediction_model.joblib')

['delay_prediction_model.joblib']